# PA2 - Judging: DPO vs SFT with a Kaggle model

Task 7. Reads the saved pairwise files from `01_pipeline_kaggle.ipynb` and judges
them with a Kaggle-hosted model. **No model outputs are generated here**, which
is what the manual requires of a judge notebook.

### This must be a benchmark notebook

`kaggle_benchmarks` is not available in an ordinary notebook. Verified on a GPU
notebook on this exact image:

```text
kaggle_benchmarks NOT importable here: No module named 'kaggle_benchmarks'
```

So create this notebook from **<https://www.kaggle.com/benchmarks/tasks/new>**,
which gives it the model-proxy environment. Pasting these cells into a normal
notebook will fail at the first judge call.

- **Accelerator**: none needed - this is all API calls
- **Internet**: `On`
- **Input**: the output of `01_pipeline_kaggle.ipynb`
- Your account must be phone-verified for LLM quota (yours is)

### Built to be restarted

The manual is explicit that fast request loops get denied, so the loop is slow
and restartable:

| Requirement | How |
|---|---|
| Small batches | 5 pairs per request |
| Sleep between attempts | >= 5 seconds |
| Retry | up to 3 attempts, then logged |
| Checkpoint after each batch | verdicts appended to JSONL immediately |
| Resume | `--reuse_judgments` skips pairs already decided |
| Failure log | separate `*_failures.jsonl` |
| Summary | win/loss/tie/unknown/failure rates |

Rerunning after a death re-judges nothing that already has a verdict, so no
quota is wasted.

## 1. Settings

In [ ]:
# Attach the output of 01_pipeline_kaggle.ipynb and point this at it.
PIPELINE_OUTPUT = "/kaggle/input/pa2-pipeline-run1"
PROJECT_DATASET = "/kaggle/input/pa2-small-model-alignment"
PROJECT_DIR = "/kaggle/working/small-model-alignment"

JUDGE_MODEL = "google/gemini-3.6-flash"
BATCH_SIZE = 5
SLEEP_SECONDS = 5
MAX_RETRIES = 3
BACKEND = "auto"       # "dry_run" exercises the plumbing without spending calls
PILOT_PAIRS = 10

RUN_PILOT = True
RUN_DPO_UF = True
RUN_DPO_HH = True
RUN_PERSONA = True

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"


def find_project(dataset_dir, work_dir):
    """Copy the project out of the read-only input dataset into /kaggle/working.

    It has to be writable: every stage writes under ``outputs/`` next to the code.

    The search is recursive because Kaggle does not guarantee where a dataset
    mounts - it has appeared at ``/kaggle/input/<slug>/`` and at
    ``/kaggle/input/datasets/<owner>/<slug>/``. Rather than guess the layout,
    look for the package itself.
    """
    target = Path(work_dir)
    if (target / "smalign").is_dir():
        print(f"using project already at {target}")
        return target

    roots = []
    if dataset_dir and Path(dataset_dir).exists():
        roots.append(Path(dataset_dir))
    if Path("/kaggle/input").is_dir():
        roots.append(Path("/kaggle/input"))
    if Path("/kaggle/working").is_dir():
        roots.append(Path("/kaggle/working"))

    for root in roots:
        for marker in sorted(root.rglob("smalign/__init__.py")):
            candidate = marker.parent.parent
            if (candidate / "submission_adapter.py").is_file():
                print(f"found project at {candidate}")
                shutil.copytree(candidate, target, dirs_exist_ok=True)
                print(f"copied -> {target}")
                return target

    # Nothing found: show what is actually mounted, which is the one thing
    # that makes this fixable without another round trip.
    listing = []
    for root in roots:
        listing.append(f"  {root}/")
        for path in sorted(root.rglob("*"))[:60]:
            depth = len(path.relative_to(root).parts)
            if depth <= 3:
                listing.append(f"    {path.relative_to(root)}{'/' if path.is_dir() else ''}")
    raise FileNotFoundError(
        "Could not find the project (looked for smalign/__init__.py).\n"
        "Attach the dataset holding it: Add Input -> Datasets -> your dataset.\n"
        "What is mounted:\n" + ("\n".join(listing) if listing else "  (nothing)")
    )


def restore_previous(prev, project, subdirs=("outputs", "data/processed", "eval_sets")):
    """Bring a previous run's artifacts in so finished stages are skipped.

    This is what makes a run resumable across sessions: the ledger, the
    checkpoints and the resume files come back, and the pipeline continues where
    it stopped instead of starting over.

    Like ``find_project``, the lookup is recursive. A kernel's output mounts at a
    path Kaggle chooses, and the project sits one level inside it, so the
    directories are located by searching rather than by assuming a layout.
    """
    if not prev:
        print("PREV_OUTPUT not set - starting fresh")
        return

    prev = Path(prev)
    if not prev.exists():
        # The configured path is wrong or Kaggle mounted it elsewhere. Rather
        # than silently start a six-hour run from zero, look for a previous
        # run's ledger anywhere under /kaggle/input and use that.
        print(f"PREV_OUTPUT {prev} does not exist - searching /kaggle/input for a previous run")
        found = None
        if Path("/kaggle/input").is_dir():
            for marker in sorted(Path("/kaggle/input").rglob("outputs/pipeline_state.json")):
                found = marker.parent.parent
                break
        if found is None:
            print("  no previous run found; starting fresh")
            print("  available inputs:",
                  [str(p) for p in Path("/kaggle/input").glob("*")]
                  if Path("/kaggle/input").is_dir() else "none")
            return
        prev = found
        print(f"  using {prev}")

    restored = []
    for sub in subdirs:
        source = None
        direct = prev / sub
        if direct.is_dir():
            source = direct
        else:
            # Find it anywhere underneath, preferring the shallowest match.
            leaf = Path(sub).name
            matches = [m for m in prev.rglob(leaf) if m.is_dir() and m.match(f"*/{sub}")]
            if not matches:
                matches = [m for m in prev.rglob(leaf) if m.is_dir()]
            if matches:
                source = min(matches, key=lambda m: len(m.parts))
        if source:
            destination = Path(project) / sub
            destination.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, destination, dirs_exist_ok=True)
            restored.append(f"{sub} <- {source}")

    for line in restored:
        print("restored", line)
    if not restored:
        print(f"nothing to restore from {prev}")
    else:
        ledger_file = Path(project) / "outputs" / "pipeline_state.json"
        if ledger_file.exists():
            state = json.loads(ledger_file.read_text())
            done = [k for k, v in state.get("stages", {}).items() if v.get("status") == "completed"]
            print(f"ledger restored: {len(done)} stage(s) already complete -> {', '.join(done) or 'none'}")


def disk_report(root="/kaggle/working"):
    try:
        total, used, free = shutil.disk_usage(root)
    except OSError:
        return
    print(f"disk {root}: {used/2**30:.1f} GiB used, {free/2**30:.1f} GiB free of {total/2**30:.1f}")
    rows = []
    for path in sorted(Path(root).rglob("*")):
        if path.is_dir() and path.name in {"best", "final", "adapter"}:
            rows.append((sum(f.stat().st_size for f in path.rglob("*") if f.is_file()), path))
        elif path.is_file() and path.name == "resume.pt":
            rows.append((path.stat().st_size, path))
    for size, path in sorted(rows, reverse=True)[:12]:
        print(f"  {size/2**30:6.2f} GiB  {path}")

In [ ]:
project = find_project(PROJECT_DATASET, PROJECT_DIR)
%cd {project}
sys.path.insert(0, str(project))

# Bring in the pairwise files and generations produced by notebook 1.
restore_previous(PIPELINE_OUTPUT, project, subdirs=("outputs", "eval_sets"))

from smalign.pipeline import Ledger, print_ledger, run_stage
from smalign.utils import read_json, read_jsonl

ledger = Ledger("outputs/judge_state.json")
EVAL = "outputs/eval"

# Regenerated every run: new judgments mean the summary tables change.
ALWAYS_RERUN = {"summarize"}

def stage(name, command, produces=(), depends_on=(), quiet_minutes=20):
    """Judging prints after every batch, so a long silence means it is wedged."""
    return run_stage(name, command, list(produces), list(depends_on), ledger=ledger,
                     force=(name in ALWAYS_RERUN), quiet_timeout=quiet_minutes * 60)

print("\npairwise files available:")
for path in sorted(Path(EVAL).glob("*_pairs.jsonl")):
    print(f"  {len(read_jsonl(path)):>4} pairs   {path}")
print()
print_ledger(ledger)

## 2. Confirm the judge model is reachable

If `google/gemini-3.6-flash` is not offered in your environment, pick another
model from the list below, set `JUDGE_MODEL`, and record which one you used -
section 13 of the manual asks for the judge model to be named in the report.

In [ ]:
try:
    import kaggle_benchmarks as kbench
    available = sorted(kbench.llms.keys())
    print(f"{len(available)} judge models available:\n")
    for name in available:
        print(f"  {name}{'   <-- configured' if name == JUDGE_MODEL else ''}")
    if JUDGE_MODEL not in available:
        print(f"\nWARNING: {JUDGE_MODEL} is not in this list.")
        print("Set JUDGE_MODEL to one of the above and note it in the report.")
except ImportError:
    print("kaggle_benchmarks is not importable in this session.")
    print("This notebook must be created from https://www.kaggle.com/benchmarks/tasks/new")
    print("or set BACKEND = 'dry_run' to exercise the plumbing without judge calls.")
except Exception as error:
    print(f"could not list judge models: {type(error).__name__}: {error}")

## 3. Pilot

Ten pairs first. Confirms the judge returns parseable verdicts in the expected shape, and shows its stated reasons - the cheapest way to notice a judge that has misread the task.

In [ ]:
if RUN_PILOT:
    stage("pilot",
          ["python", "scripts/judge_pairwise_kaggle.py",
           "--pairs_path", f"{EVAL}/dpo_uf_vs_sft_pairs.jsonl",
           "--output_path", f"{EVAL}/pilot_judged.jsonl",
           "--judge_model", JUDGE_MODEL, "--backend", BACKEND,
           "--batch_size", str(BATCH_SIZE), "--sleep_seconds", str(SLEEP_SECONDS),
           "--max_retries", str(MAX_RETRIES), "--limit", str(PILOT_PAIRS)],
          produces=[f"{EVAL}/pilot_judged.jsonl"])

    if Path(f"{EVAL}/pilot_judged.jsonl").exists():
        pilot = read_jsonl(f"{EVAL}/pilot_judged.jsonl")
        print(f"\n{len(pilot)} pilot verdicts\n")
        for row in pilot:
            print(f"  {row['id']:<20} {row['winner']:<8} -> "
                  f"{str(row['winner_model']):<32} {row['reason'][:70]}")
        bad = [r for r in pilot if r["winner"] not in {"A", "B", "TIE"}]
        print(f"\n{len(bad)} unresolved" if bad else "\npilot is clean; continue")
        if bad:
            print("if structured output is unsupported here, add --no_schema")

## 4. UltraFeedback DPO vs SFT on AlpacaEval

300 pairs on general helpfulness: helpfulness, correctness, instruction-following, concision, formatting, and absence of hallucination or unsafe content. At 5 per request with a 5-second pause, expect 5-10 minutes.

In [ ]:
if RUN_DPO_UF:
    stage("judge_dpo_uf",
          ["python", "scripts/judge_pairwise_kaggle.py",
           "--pairs_path", f"{EVAL}/dpo_uf_vs_sft_pairs.jsonl",
           "--output_path", f"{EVAL}/dpo_uf_vs_sft_judged.jsonl",
           "--judge_model", JUDGE_MODEL, "--backend", BACKEND,
           "--batch_size", str(BATCH_SIZE), "--sleep_seconds", str(SLEEP_SECONDS),
           "--max_retries", str(MAX_RETRIES), "--reuse_judgments"],
          produces=[f"{EVAL}/dpo_uf_vs_sft_judged.summary.json"])

## 5. HH-RLHF DPO vs SFT on the helpful/harmless set

300 pairs with the safety criteria: appropriate refusal, **no over-refusal**,
privacy protection, calibrated uncertainty, and safe redirection.

Over-refusal is scored as a failure, not a virtue. The evaluation set carries 30
benign prompts precisely so a model that refuses everything cannot look safe
while being useless.

In [ ]:
if RUN_DPO_HH:
    stage("judge_dpo_hh",
          ["python", "scripts/judge_pairwise_kaggle.py",
           "--pairs_path", f"{EVAL}/dpo_hh_vs_sft_pairs.jsonl",
           "--output_path", f"{EVAL}/dpo_hh_vs_sft_judged.jsonl",
           "--judge_model", JUDGE_MODEL, "--backend", BACKEND,
           "--batch_size", str(BATCH_SIZE), "--sleep_seconds", str(SLEEP_SECONDS),
           "--max_retries", str(MAX_RETRIES), "--safety_mode", "--reuse_judgments"],
          produces=[f"{EVAL}/dpo_hh_vs_sft_judged.summary.json"])

## 6. Persona LoRA-DPO vs base

Reported separately: style alignment is a different target from helpfulness or safety, and section 14.5 forbids combining unlike evaluation sets into one headline score.

In [ ]:
if RUN_PERSONA and Path(f"{EVAL}/lora_persona_vs_base_pairs.jsonl").exists():
    stage("judge_persona",
          ["python", "scripts/judge_pairwise_kaggle.py",
           "--pairs_path", f"{EVAL}/lora_persona_vs_base_pairs.jsonl",
           "--output_path", f"{EVAL}/lora_persona_vs_base_judged.jsonl",
           "--judge_model", JUDGE_MODEL, "--backend", BACKEND,
           "--batch_size", str(BATCH_SIZE), "--sleep_seconds", str(SLEEP_SECONDS),
           "--max_retries", str(MAX_RETRIES), "--reuse_judgments"],
          produces=[f"{EVAL}/lora_persona_vs_base_judged.summary.json"])

## 7. Results

In [ ]:
stage("summarize", ["python", "scripts/summarize_results.py", "--outputs_dir", "outputs"])

In [ ]:
from smalign.judge import print_summary

for path in sorted(Path(EVAL).glob("*_judged.summary.json")):
    if path.name.startswith("pilot"):
        continue
    summary = read_json(path)
    print("#" * 78)
    print(f"# {path.name.replace('_judged.summary.json', '')}")
    print("#" * 78)
    print_summary(summary)
    print(f"retries {summary.get('retry_attempts')} | failed batches "
          f"{summary.get('failed_batches')} | tie rate {summary.get('tie_rate')} "
          f"| unknown {summary.get('unknown_rate')} | failure {summary.get('failure_rate')}")
    print()

### Per-category breakdown

On the helpful/harmless set this is where you find out whether DPO bought safety at the cost of over-refusing the benign prompts - the alignment tax, if there is one.

In [ ]:
for path in sorted(Path(EVAL).glob("*_judged.summary.json")):
    if path.name.startswith("pilot"):
        continue
    by_category = (read_json(path).get("by_category") or {})
    if not by_category:
        continue
    print("=" * 78); print(path.name.replace("_judged.summary.json", "")); print("=" * 78)
    header = f"{'category':<28} {'ties':>5} {'unresolved':>11}   wins"
    print(header); print("-" * len(header))
    for category, bucket in sorted(by_category.items()):
        wins = ", ".join(f"{m}={c}" for m, c in sorted(bucket["wins"].items()))
        print(f"{category:<28} {bucket['ties']:>5} {bucket['unresolved']:>11}   {wins}")
    print()

### Representative verdicts

The examples section 14.6 asks you to quote - including ones that went against the aggregate, which is how you notice a judge rewarding length or formatting rather than quality.

In [ ]:
def show_verdicts(judged_name, pairs_name, winner_model=None, n=2, width=500):
    jp, pp = Path(EVAL) / judged_name, Path(EVAL) / pairs_name
    if not (jp.exists() and pp.exists()):
        print(f"(missing {jp} or {pp})"); return
    judged = {r["id"]: r for r in read_jsonl(jp)}
    pairs = {r["id"]: r for r in read_jsonl(pp)}
    shown = 0
    for row_id, verdict in judged.items():
        if verdict["winner"] not in {"A", "B"}: continue
        if winner_model and verdict["winner_model"] != winner_model: continue
        pair = pairs.get(row_id)
        if not pair: continue
        print("=" * 78)
        print(f"{row_id}  winner: {verdict['winner_model']}")
        print(f"judge reason: {verdict['reason']}"); print("-" * 78)
        print("PROMPT:", pair["instruction"][:250])
        for side in ("a", "b"):
            mark = " <-- judge preferred" if verdict["winner"] == side.upper() else ""
            print("-" * 78)
            print(f"[{pair['meta'][f'{side}_model']}]{mark}\n"
                  f"{pair[f'response_{side}'][:width] or '(empty)'}")
        print()
        shown += 1
        if shown >= n: return

print("#" * 78); print("# AlpacaEval: given to DPO"); print("#" * 78)
show_verdicts("dpo_uf_vs_sft_judged.jsonl", "dpo_uf_vs_sft_pairs.jsonl",
              winner_model="qwen2.5-0.5b-dpo-uf")
print("#" * 78); print("# AlpacaEval: given to SFT (where DPO regressed)"); print("#" * 78)
show_verdicts("dpo_uf_vs_sft_judged.jsonl", "dpo_uf_vs_sft_pairs.jsonl",
              winner_model="qwen2.5-0.5b-sft")

### Failure log

Section 14.5 asks for failures, retries, and the handling of invalid judgments - those counts belong in the report even when they are zero.

In [ ]:
failures = sorted(Path(EVAL).glob("*_failures.jsonl"))
if not failures:
    print("no failure logs: every batch resolved within its retry budget")
for path in failures:
    rows = read_jsonl(path)
    print(f"{path}: {len(rows)} unresolved")
    for row in rows[:10]:
        print(f"  {row['id']}  after {row['attempts']} attempts: {str(row.get('error'))[:120]}")

## 8. Archive

In [ ]:
print_ledger(ledger)

!rm -f /kaggle/working/judgments.zip
!cd /kaggle/working/small-model-alignment && zip -q -r /kaggle/working/judgments.zip \
    outputs/eval outputs/results_summary.json outputs/results_summary.md 2>/dev/null
!ls -la /kaggle/working/judgments.zip 2>/dev/null

print("\noutputs/results_summary.md holds the tables for the report.")
for pattern in ("*_judged.jsonl", "*_judged.summary.json", "*_failures.jsonl"):
    for path in sorted(Path(EVAL).glob(pattern)):
        print(f"  {path}")

---

## Writing the report

`report/REPORT_TEMPLATE.md` has the seven required sections with the numbers to
fill in and where each comes from.

| Need | Source |
|---|---|
| Configuration and judging tables | `outputs/results_summary.md` |
| Training plots | `outputs/plots/*.png` (from notebook 1) |
| Values actually used, not guide values | `outputs/<run>/run_config.json` |
| Wins, ties, failures, judge model | `outputs/eval/*_judged.summary.json` |
| Per-category safety breakdown | the `by_category` block above |

The report asks for the configuration of the **submitted** checkpoints and for
every deviation from the guide recipe to be named with its reason. The template
already lists the deviations built into this implementation.